In [ ]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os

import sys

sys.path.append("../..")

# Modern import pattern - unified simulation method with strategy pattern
from src import Multicolour_Simulation_Functions
from src.Multicolour_Simulation_Functions import FittingStrategy, SimulationConfig

# Additional required components not integrated into main simulation class
from src import PlottingFunctions
from src import SpectralFunctions
from src import MaskFunctions

# Create main simulation instance (contains IO, PSF, sCMOS, ImageAnalysis dependencies)
MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

# Access integrated components through MSF
IO = MSF.io
I_AF = MSF.image_analysis
sCMOS = MSF.scmos
PSF = MSF.psf

# Create instances of non-integrated components
plotter = PlottingFunctions.Plotter()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()

In [ ]:
fretfluors = pl.read_csv(
    "/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/FRETFluors/FluorescenceSpectra_FRETfluors_Normalised.csv"
)

In [ ]:
R, G, B, wavelength = S_F.getpixelefficiency()
minwl = np.argmin(np.abs(wavelength - fretfluors["wavelength"].min()))
maxwl = np.argmin(np.abs(wavelength - fretfluors["wavelength"].max()))
wavelength = wavelength[minwl : maxwl + 1]
R = R[minwl : maxwl + 1]
G = G[minwl : maxwl + 1]
B = B[minwl : maxwl + 1]
pixel_QYs = np.vstack([B, G, R])

In [ ]:
spectra = fretfluors.to_numpy()[:, 1:]

In [ ]:
weighted_wavelengths = wavelength * spectra.T
average_wavelengths = np.trapz(y=weighted_wavelengths.T, x=wavelength, axis=0)

In [ ]:
pixel_efficiencies = np.dot(spectra.T, pixel_QYs.T)
pixel_efficiencies = pixel_efficiencies.T / np.sum(pixel_efficiencies, axis=1)
pixel_efficiencies = pixel_efficiencies.T

In [ ]:
fig, axs = plotter.two_column_plot(nrows=2, heightratio=[1, 1])

axs = plotter.ternary_scatter_plot(
    fig,
    axs,
    R=pixel_efficiencies[:, -1],
    G=pixel_efficiencies[:, 1],
    B=pixel_efficiencies[:, 0],
    colours="black",
    s=5,
)

plt.savefig(
    "/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Talks+Posters/Subgroup/20250911/fig/multicolour/FRETfluors/Separability_FFs.svg",
    dpi=600,
    format="svg",
)
plt.show()